# Sign calculator

In [ ]:
from time import sleep
from pynq.overlays.base import BaseOverlay

# Convert switches + sign into value (-3 to 3)
def get_value(sw0, sw1, sign):
    base = (sw1 << 1) | sw0   # 0 to 3

    if sign:
        return -base
    else:
        return base

# Display result on LEDs
def display(base, result):
    # Limit range to fit LEDs
    if result < -7:
        result = -7
    if result > 7:
        result = 7

    # Sign on LED3
    if result < 0:
        base.leds[3].on()
        result = abs(result)
    else:
        base.leds[3].off()

    # Magnitude on LED0–LED2
    for i in range(3):
        if result & (1 << i):
            base.leds[i].on()
        else:
            base.leds[i].off()

def main():
    base = BaseOverlay('base.bit')

    mode = 0        # 0 = A, 1 = B
    operation = 0   # 0=ADD,1=SUB,2=MUL,3=DIV
    A = 0
    B = 0
    sign = 0        # 0=positive, 1=negative

    prev = [0, 0, 0, 0]

    print("Calculator Started (-3 to 3)")

    while True:
        curr = [base.buttons[i].read() for i in range(4)]

        # BTN0 → Toggle sign
        if curr[0] == 1 and prev[0] == 0:
            sign = 1 - sign
            print(f"Sign = {'-' if sign else '+'}")

        # BTN1 → Capture value
        if curr[1] == 1 and prev[1] == 0:
            sw0 = base.switches[0].read()
            sw1 = base.switches[1].read()
            value = get_value(sw0, sw1, sign)

            if mode == 0:
                A = value
                print(f"A = {A}")
            else:
                B = value
                print(f"B = {B}")

            # Toggle mode automatically after capture
            mode = 1 - mode
            print(f"Next input: {'A' if mode == 0 else 'B'}")

        # BTN2 → Change operation
        if curr[2] == 1 and prev[2] == 0:
            operation = (operation + 1) % 4
            ops = ["ADD", "SUB", "MUL", "DIV"]
            print(f"Operation = {ops[operation]}")

        # BTN3 → Compute
        if curr[3] == 1 and prev[3] == 0:
            if operation == 0:
                result = A + B
                op = "+"
            elif operation == 1:
                result = A - B
                op = "-"
            elif operation == 2:
                result = A * B
                op = "*"
            elif operation == 3:
                if B != 0:
                    result = int(A / B)
                else:
                    print("Divide by zero!")
                    result = 0
                op = "/"

            print(f"{A} {op} {B} = {result}")
            display(base, result)

        prev = curr.copy()
        sleep(0.05)

print("Code starting...")
main()